# M03A · Classes and Objects in Bioinformatics
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2024-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M03_OOP/M03A_Classes_and_Objects.ipynb)

⏱ ~90 min · Difficulty: ★★☆ · Prerequisites: Module 1 (Core Python, Functions, Dictionaries)

---


## 1. Learning Outcomes

By the end of this notebook, you will be able to:
1. **Contrast** procedural code with object-oriented design in the context of biological sequence modeling.
2. **Define** Python classes with constructors (`__init__`), instance attributes, and nucleotide/amino-acid alphabet validation.
3. **Distinguish** between instance attributes and class attributes, identifying the architectural risks of shared mutable state.
4. **Implement** instance methods that calculate sequence metrics (GC content, molecular weight) and return new immutable object instances.
5. **Implement** special "dunder" methods (`__str__`, `__repr__`, `__len__`, `__eq__`, `__add__`, `__getitem__`) to integrate custom biological types seamlessly with Python syntax.
6. **Design** alternative constructors using `@classmethod` and standalone utility functions using `@staticmethod`.

---


## 2. Prerequisites

Before starting this notebook, ensure you understand:
- Python functions, keyword arguments, and return values.
- Dictionaries, lists, and string slicing.
- Basic molecular biology: DNA nucleotides, transcription, reverse complementation, and codon structures.
- Completed **Module 1 (Basics of Python)** and **Module 2 (File Handling)**.

---


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

# Detect whether we are running in Google Colab or locally
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root
    ROOT = Path.cwd().resolve()
    while not (ROOT / "data").exists() and ROOT.parent != ROOT:
        ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M03A"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python interpreter : {sys.version.split()[0]}")
print(f"Repository ROOT    : {ROOT.resolve()}")
print(f"DATA_DIR exists    : {DATA_DIR.exists()} ({DATA_DIR})")
print(f"OUT_DIR ready      : {OUT_DIR.exists()} ({OUT_DIR})")


## 4. Concept Sections

### 4.1 Concept 1: Procedural vs Object-Oriented Programming

In computational biology, biological entities possess both **state** (data) and **behavior** (operations):
- A **Gene** has an accession ID, genomic coordinates, a strand, and a sequence (data). It can be transcribed, spliced, or mutated (behavior).
- A **Protein** has an amino acid sequence and a 3D structure (data). It can fold, bind ligands, or catalyze enzymatic reactions (behavior).

#### The Procedural Approach
In procedural programming, data and functions are detached. Data is stored in loose dictionaries, tuples, or parallel lists, and passed into independent functions:
```python
# Procedural style
gene_id = "TP53"
seq = "ATGGAGGAG..."
strand = "+"

def calc_gc(s): ...
def get_revcomp(s): ...
```
**Problems with procedural design at scale:**
1. **Data Desynchronization**: Slicing or mutating `seq` doesn't update associated coordinates or length fields.
2. **Type Safety**: Any function accepting a string can be accidentally passed non-DNA text, garbage characters, or protein sequences.
3. **Namespace Clutter**: You end up with dozens of prefixed functions: `dna_gc()`, `rna_gc()`, `dna_translate()`.

#### The Object-Oriented Approach
Object-Oriented Programming (OOP) bundles state and behavior into a cohesive entity called an **Object**, instantiated from a blueprint called a **Class**.


In [ ]:
# Worked Example 4.1: Procedural vs OOP comparison
# Procedural style: loose tuple and external functions
proc_gene = ("TP53", "ATGGAGGAGCCGCAGTCAGAT", 17)

def proc_gc(gene_tuple):
    seq = gene_tuple[1]
    return (seq.count("G") + seq.count("C")) / len(seq) * 100.0

print(f"Procedural GC: {proc_gc(proc_gene):.2f}%")

# OOP style: cohesive class
class GeneDemo:
    def __init__(self, name: str, seq: str, chrom: int):
        self.name = name
        self.seq = seq.upper()
        self.chrom = chrom

    def gc_content(self) -> float:
        return (self.seq.count("G") + self.seq.count("C")) / len(self.seq) * 100.0

oop_gene = GeneDemo("TP53", "ATGGAGGAGCCGCAGTCAGAT", 17)
print(f"OOP GC       : {oop_gene.gc_content():.2f}% for {oop_gene.name} on chr{oop_gene.chrom}")


#### ✏️ Try it 4.1: Instantiate and Query an Object
Instantiate `GeneDemo` with name `"BRCA1"`, sequence `"ATGGATTTATCTGCT"`, and chromosome `17`. Verify its GC content.


In [ ]:
# Try it 4.1: GeneDemo test
brca1 = GeneDemo("BRCA1", "ATGGATTTATCTGCT", 17)
gc_val = brca1.gc_content()

print(f"{brca1.name} GC: {gc_val:.2f}%")
assert brca1.name == "BRCA1"
assert brca1.chrom == 17
assert round(gc_val, 2) == 33.33
print("Try it 4.1 passed ✔")


#### ⚠️ Common Mistakes in OOP
- **Passing `self` explicitly**: Writing `oop_gene.gc_content(oop_gene)`. Python automatically passes the instance as the first argument (`self`).
- **Over-engineering simple scripts**: Wrapping a 5-line one-off text manipulation script into 4 classes is unnecessary complexity. OOP shines when entities have persistent state and multiple interacting operations.


### 4.2 Concept 2: Class Blueprint & Object Instantiation

- **Class**: The blueprint or template (e.g., `class DNASequence:`). Defined once in code.
- **Instance (Object)**: A specific concrete occurrence created from the blueprint (e.g., `seq1 = DNASequence(...)`).
- Each instance occupies its own distinct location in memory, verifiable using Python's built-in `id(obj)`.

#### Naming Conventions (PEP 8)
- Class names use **PascalCase** (e.g., `DNASequence`, `RestrictionEnzyme`, `GenomicInterval`).
- Method and attribute names use **snake_case** (e.g., `gc_content`, `reverse_complement`, `start_pos`).


In [ ]:
# Worked Example 4.2: Defining a basic Gene class
class Gene:
    """Represents a genomic gene locus."""
    def __init__(self, name: str, chrom: str, start: int, end: int, strand: str = "+"):
        self.name = name
        self.chrom = chrom
        self.start = start
        self.end = end
        self.strand = strand

    def length(self) -> int:
        """Return genomic length in base pairs."""
        return abs(self.end - self.start) + 1

# Instantiate two distinct gene objects
gene_tp53 = Gene("TP53", "chr17", 7668402, 7687550, strand="-")
gene_ins  = Gene("INS", "chr11", 2159779, 2161209, strand="-")

print(f"TP53 length: {gene_tp53.length():,} bp | Memory ID: {hex(id(gene_tp53))}")
print(f"INS  length: {gene_ins.length():,} bp | Memory ID: {hex(id(gene_ins))}")
assert gene_tp53 is not gene_ins


#### ✏️ Try it 4.2: Gene Length Calculation
Instantiate a gene `"lacZ"` spanning coordinates `1283` to `4357` on chromosome `"chr1"`, and assert its length is `3075` bp.


In [ ]:
# Try it 4.2: lacZ gene instance
lacz = Gene("lacZ", "chr1", 1283, 4357, strand="+")
assert lacz.length() == 3075, f"Expected 3075, got {lacz.length()}"
print(f"lacZ length: {lacz.length()} bp ✔")


#### ⚠️ Common Mistakes with Instantiation
- **Forgetting parentheses during instantiation**: Writing `my_gene = Gene` assigns the class blueprint itself to the variable, rather than creating an instance! You must write `my_gene = Gene(...)`.
- **Violating PEP 8 naming**: Naming classes `dna_sequence` makes code hard to distinguish from instance variables.


### 4.3 Concept 3: The Constructor `__init__`, `self`, and Data Validation

When you instantiate a class with `Gene("TP53", ...)`, Python performs two steps under the hood:
1. `__new__()`: Allocates a new blank object in memory.
2. `__init__()`: The **initializer/constructor**, which binds attributes to `self`.

#### What is `self`?
`self` represents the specific instance being initialized or called. It is explicitly declared as the first parameter in all instance methods, but automatically supplied by Python when called on an instance (`instance.method()`).

#### Validation in `__init__`
One of the greatest advantages of OOP is enforcing **data invariants**. If a DNA sequence contains invalid characters (e.g. `'Z'`, `'1'`), `__init__` can immediately raise a `ValueError`, preventing silent bugs downstream.


In [ ]:
# Worked Example 4.3: Validating biological invariants in __init__
class ValidatedDNA:
    VALID_BASES = {"A", "C", "G", "T", "N"}

    def __init__(self, seq_id: str, sequence: str):
        self.seq_id = seq_id
        clean_seq = sequence.strip().upper()
        
        # Enforce validation
        invalid_bases = set(clean_seq) - self.VALID_BASES
        if invalid_bases:
            raise ValueError(f"Invalid nucleotide base(s) detected: {invalid_bases} in '{clean_seq[:20]}'")
        
        self.sequence = clean_seq

# Test valid sequence
v_dna = ValidatedDNA("seq1", "atgcgtacgt")
print(f"Validated sequence: {v_dna.seq_id} -> {v_dna.sequence}")

# Test invalid sequence
try:
    bad_dna = ValidatedDNA("bad_seq", "ATGCZ123")
except ValueError as e:
    print(f"Caught expected error: {e}")


#### ✏️ Try it 4.3: Assert Constructor Validation
Verify that instantiating `ValidatedDNA` with `"ATG123"` raises `ValueError`.


In [ ]:
# Try it 4.3: Test validation assertion
raised_error = False
try:
    ValidatedDNA("test", "ATG123")
except ValueError:
    raised_error = True

assert raised_error, "Should raise ValueError on numeric characters in DNA!"
print("Try it 4.3 passed ✔")


#### ⚠️ Common Mistakes in `__init__`
- **Forgetting `self.`**: Writing `sequence = clean_seq` instead of `self.sequence = clean_seq`. Without `self.`, `sequence` remains a temporary local variable and is discarded when `__init__` returns.
- **Returning values from `__init__`**: Constructors in Python must always return `None`. Writing `return self` or `return True` raises `TypeError: __init__() should return None`.


### 4.4 Concept 4: Instance Attributes vs Class Attributes

In Python, attributes can exist at two different scopes:

1. **Instance Attributes**:
   - Defined inside methods using `self.attr = value`.
   - Stored in the individual instance's dictionary (`instance.__dict__`).
   - Unique to each instance. Modifying it on one instance does not affect any other instance.

2. **Class Attributes**:
   - Defined directly inside the `class` block, outside any method.
   - Stored in the class dictionary (`Class.__dict__`).
   - Shared across **all** instances of that class.
   - Ideal for constants, default configurations, or lookup tables (e.g. `GENETIC_CODE`).

#### The Mutable Class Attribute Trap!
If a class attribute is a **mutable object** (like a `list` or `dict`), mutating it in-place via an instance (`self.shared_list.append(...)`) mutates it for **every instance** of that class!


In [ ]:
# Worked Example 4.4: Instance vs Class attributes
class OrganismRecord:
    # Class attribute: shared default kingdom across all instances
    DEFAULT_KINGDOM = "Metazoa"

    def __init__(self, species_name: str, common_name: str):
        # Instance attributes: unique to each instance
        self.species_name = species_name
        self.common_name = common_name

human = OrganismRecord("Homo sapiens", "Human")
mouse = OrganismRecord("Mus musculus", "House mouse")

print(f"Human kingdom : {human.DEFAULT_KINGDOM} ({human.common_name})")
print(f"Mouse kingdom : {mouse.DEFAULT_KINGDOM} ({mouse.common_name})")

# Modifying class attribute via class affects all instances
OrganismRecord.DEFAULT_KINGDOM = "Animalia"
print(f"Updated Human kingdom : {human.DEFAULT_KINGDOM}")
print(f"Updated Mouse kingdom : {mouse.DEFAULT_KINGDOM}")


#### ✏️ Try it 4.4: Inspect `__dict__`
Inspect `human.__dict__` and confirm that `"species_name"` is an instance attribute, while `"DEFAULT_KINGDOM"` is absent from `human.__dict__` (lives in `OrganismRecord.__dict__`).


In [ ]:
# Try it 4.4: Attribute scope inspection
assert "species_name" in human.__dict__
assert "DEFAULT_KINGDOM" not in human.__dict__
assert "DEFAULT_KINGDOM" in OrganismRecord.__dict__
print("Try it 4.4 passed ✔")


#### ⚠️ Common Mistakes with Class Attributes
- **Accidental Attribute Shadowing**: If you write `human.DEFAULT_KINGDOM = "Custom"`, Python does NOT change the class attribute. Instead, it creates a new instance attribute named `DEFAULT_KINGDOM` on `human`, shadowing the class attribute!
- **Mutable Class Attributes**: Defining `class Sample: reads = []` means every sample will share the exact same list of reads!


### 4.5 Concept 5: Instance Methods & The Immutability Pattern

An **Instance Method** is a function defined inside a class that accepts `self` as its first parameter and operates on the instance's state.

#### The Immutability Pattern for Biological Sequences
When you call `reverse_complement()` or `transcribe()` on a sequence, should it mutate the sequence in-place, or return a brand-new object?

In modern bioinformatics software (including Biopython `Bio.Seq`), biological sequences are treated as **immutable**:
- Mutating an existing sequence in-place creates severe bugs if other parts of your program (e.g. alignment tables, variant databases) reference that sequence.
- Returning a new instance (`return DNASequence(rev_comp_seq)`) preserves state integrity and enables elegant method chaining:
  ```python
  protein = dna.transcribe().translate()
  ```


In [ ]:
# Worked Example 4.5: Immutable methods on DNASequence
class DNASequence:
    TRANS_TABLE = str.maketrans("ACGTNacgtn", "TGCANtgcan")

    def __init__(self, seq_id: str, seq: str):
        self.seq_id = seq_id
        self.seq = seq.strip().upper()

    def gc_content(self) -> float:
        """Calculate GC percentage."""
        if not self.seq:
            return 0.0
        gc = self.seq.count("G") + self.seq.count("C")
        return round((gc / len(self.seq)) * 100.0, 2)

    def reverse_complement(self) -> "DNASequence":
        """Return a NEW DNASequence object containing the reverse complement."""
        rc_seq = self.seq.translate(self.TRANS_TABLE)[::-1]
        return DNASequence(f"{self.seq_id}_rc", rc_seq)

    def transcribe(self) -> str:
        """Transcribe DNA to RNA string."""
        return self.seq.replace("T", "U")

# Demonstration of immutability
dna1 = DNASequence("probe_1", "ATGCGT")
dna1_rc = dna1.reverse_complement()

print(f"Original sequence  : {dna1.seq_id} -> {dna1.seq}")
print(f"New RC sequence    : {dna1_rc.seq_id} -> {dna1_rc.seq}")
assert dna1.seq == "ATGCGT", "Original sequence must remain unchanged!"


#### ✏️ Try it 4.5: Method Chaining Check
Verify that calling `dna1.reverse_complement().reverse_complement()` yields the identical sequence to `dna1.seq`.


In [ ]:
# Try it 4.5: Double reverse complement
double_rc = dna1.reverse_complement().reverse_complement()
assert double_rc.seq == dna1.seq, f"Expected {dna1.seq}, got {double_rc.seq}"
print("Try it 4.5 passed ✔")


#### ⚠️ Common Mistakes in Instance Methods
- **Forgetting `return`**: Defining a method that computes a value but forgets `return result` will silently return `None`.
- **Modifying `self.seq` in-place**: Changing `self.seq = ...` inside a calculation method breaks the immutability contract.


### 4.6 Concept 6: Special "Dunder" (Magic) Methods

In Python, classes integrate seamlessly with language operators through **dunder methods** (double underscore methods). Implementing these methods allows your custom classes to support Python's built-in functions and operators:

| Dunder Method | Triggered By | Biological Meaning |
|---|---|---|
| `__str__(self)` | `print(obj)`, `str(obj)` | Readable summary for humans/reports |
| `__repr__(self)` | Interactive REPL, `repr(obj)` | Unambiguous representation for debugging |
| `__len__(self)` | `len(obj)` | Sequence length in base pairs |
| `__eq__(self, other)` | `obj1 == obj2` | Equivalence of sequence content |
| `__add__(self, other)` | `obj1 + obj2` | Ligation / concatenation of sequences |
| `__getitem__(self, idx)` | `obj[start:end]` | Subsequence slicing |


In [ ]:
# Worked Example 4.6: Comprehensive Dunder Methods for DNA
class RichDNASequence:
    def __init__(self, seq_id: str, seq: str):
        self.seq_id = seq_id
        self.seq = seq.strip().upper()

    def __str__(self) -> str:
        preview = self.seq[:15] + "..." if len(self.seq) > 15 else self.seq
        return f">{self.seq_id} ({len(self.seq)} bp): {preview}"

    def __repr__(self) -> str:
        return f"RichDNASequence(seq_id={repr(self.seq_id)}, seq={repr(self.seq)})"

    def __len__(self) -> int:
        return len(self.seq)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, RichDNASequence):
            return False
        return self.seq == other.seq

    def __add__(self, other: "RichDNASequence") -> "RichDNASequence":
        if not isinstance(other, RichDNASequence):
            raise TypeError("Can only concatenate RichDNASequence with another RichDNASequence")
        return RichDNASequence(f"{self.seq_id}+{other.seq_id}", self.seq + other.seq)

    def __getitem__(self, item) -> str:
        return self.seq[item]

s1 = RichDNASequence("oligo1", "ATGCGT")
s2 = RichDNASequence("oligo2", "ACGTAC")

print(f"str(s1)  : {s1}")
print(f"repr(s1) : {repr(s1)}")
print(f"len(s1)  : {len(s1)} bp")
print(f"s1 + s2  : {s1 + s2}")
print(f"s1[0:3]  : {s1[0:3]}")
print(f"s1 == s2 : {s1 == s2}")


#### ✏️ Try it 4.6: Test Dunder Methods
Assert that `len(s1 + s2) == 12` and that `s1[0:4] == "ATGC"`.


In [ ]:
# Try it 4.6: Dunder assertions
ligation = s1 + s2
assert len(ligation) == 12, f"Expected 12 bp, got {len(ligation)}"
assert s1[0:4] == "ATGC"
assert s1 != s2
print("Try it 4.6 passed ✔")


#### ⚠️ Common Mistakes with Dunder Methods
- **Returning non-strings from `__str__` or `__repr__`**: Python strictly requires a string return. Returning an int or tuple raises `TypeError`.
- **Returning non-ints from `__len__`**: Returning a float or string from `__len__` raises `TypeError: 'float' object cannot be interpreted as an integer`.
- **Implementing `__eq__` without `isinstance` check**: Comparing with a non-sequence object can cause `AttributeError: 'int' object has no attribute 'seq'`.


### 4.7 Concept 7: `@classmethod` & `@staticmethod`

Methods are not restricted to operating on instance state:

1. **`@classmethod`**:
   - Accepts `cls` (the class itself) as its first parameter instead of `self`.
   - **Primary Use Case**: Factory methods / alternative constructors (e.g. creating an object from a FASTA string, GenBank file, or JSON record).
2. **`@staticmethod`**:
   - Accepts neither `self` nor `cls`.
   - Behaves like a plain function, but belongs inside the class namespace for logical organization.
   - **Primary Use Case**: Utility helpers, validation checks, or coordinate transformations.


In [ ]:
# Worked Example 4.7: Class methods and static methods
class BioSeqWithFactories:
    def __init__(self, seq_id: str, seq: str):
        self.seq_id = seq_id
        self.seq = seq.upper()

    @classmethod
    def from_fasta_string(cls, fasta_str: str) -> "BioSeqWithFactories":
        """Factory method: parses a single-record FASTA string."""
        lines = [l.strip() for l in fasta_str.strip().splitlines() if l.strip()]
        if not lines or not lines[0].startswith(">"):
            raise ValueError("Invalid FASTA: header must start with '>'")
        seq_id = lines[0][1:].split()[0]
        seq = "".join(lines[1:])
        return cls(seq_id, seq)

    @staticmethod
    def is_valid_dna(seq_str: str) -> bool:
        """Static utility: validate DNA alphabet without instantiating an object."""
        return set(seq_str.upper()).issubset({"A", "C", "G", "T", "N"})

# Using static method directly on class
print("Is 'ATGC' valid DNA? :", BioSeqWithFactories.is_valid_dna("ATGC"))
print("Is 'ATGCZ' valid DNA?:", BioSeqWithFactories.is_valid_dna("ATGCZ"))

# Using classmethod factory to create an instance
raw_fasta = ">sample_demo Putative operon sequence\nATGCGTAC\nGTTAGCTA\n"
instance = BioSeqWithFactories.from_fasta_string(raw_fasta)
print(f"Factory created: {instance.seq_id} -> {instance.seq}")


#### ✏️ Try it 4.7: Factory Constructor Test
Verify that `BioSeqWithFactories.from_fasta_string` correctly parses an inline FASTA string and creates a valid instance.


In [ ]:
# Try it 4.7: Factory test
rec = BioSeqWithFactories.from_fasta_string(">TEST_01 Description\nGGCCAATT\n")
assert rec.seq_id == "TEST_01"
assert rec.seq == "GGCCAATT"
print("Try it 4.7 passed ✔")


#### ⚠️ Common Mistakes with Decorators
- **Confusing `self` and `cls`**: Writing `self` in a `@classmethod` parameter list works syntactically, but violates convention and confuses developers. Always name the first parameter `cls`.
- **Calling `self` inside `@staticmethod`**: A `@staticmethod` has no access to `self` or `cls`. Attempting to access `self.seq` will raise `NameError: name 'self' is not defined`.


---
## 5. Practice Exercises

Apply your OOP skills to solve these bioinformatics exercises:
- **Level 1 (★)**: Basic classes, initialization, and methods (`Gene`, `Protein`).
- **Level 2 (★★)**: Spliced transcript modeling, operator overloading, and factory constructors.
- **Level 3 (★★★)**: Restriction enzyme digestion simulator.

---


### Ex 3A.1 (★) — The `Gene` Model Class

**Problem Statement:**  
Create a class `Gene` that models a genomic gene feature:
1. `__init__(self, name: str, chromosome: str, start: int, end: int, strand: str = "+")`:
   - Store all arguments as instance attributes.
   - Raise `ValueError` if `strand` is not `"+"` or `"-"`.
   - Raise `ValueError` if `start > end`.
2. Method `length(self) -> int`:
   - Returns genomic span: `end - start + 1`.
3. Method `__str__(self) -> str`:
   - Returns formatted string: `"{name} on {chromosome}:{start}-{end} ({strand})"`.

- **Input:** Instantiation parameters.
- **Output:** Fully functional `Gene` object.


In [ ]:
class Gene:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 3A.1
g = Gene("TP53", "chr17", 7668402, 7687550, strand="-")
assert g.name == "TP53"
assert g.length() == 19149
assert str(g) == "TP53 on chr17:7668402-7687550 (-)"

# Test validations
try:
    Gene("BAD", "chr1", 500, 100)
    assert False, "Should raise ValueError on start > end"
except ValueError:
    pass

try:
    Gene("BAD2", "chr1", 100, 500, strand="forward")
    assert False, "Should raise ValueError on invalid strand"
except ValueError:
    pass

print("Ex 3A.1 passed ✔")


### Ex 3A.2 (★) — The `Protein` Class

**Problem Statement:**  
Create a class `Protein` that models a polypeptide chain:
1. Class attribute `AMINO_ACID_WEIGHTS`:
   - Dictionary of average residue molecular weights (in Daltons, after dehydration):
     `{'A': 71.08, 'C': 103.14, 'D': 115.09, 'E': 129.12, 'F': 147.18, 'G': 57.05, 'H': 137.14, 'I': 113.16, 'K': 128.17, 'L': 113.16, 'M': 131.20, 'N': 114.10, 'P': 97.12, 'Q': 128.13, 'R': 156.19, 'S': 87.08, 'T': 101.11, 'V': 99.13, 'W': 186.21, 'Y': 163.18}`
2. `__init__(self, protein_id: str, sequence: str)`:
   - Validates that every character in `sequence` belongs to the 20 standard amino acids.
   - Raises `ValueError` if any non-standard character is found.
   - Stores uppercase sequence.
3. Method `molecular_weight(self) -> float`:
   - Calculates molecular weight: sum of residue weights + $18.015\text{ Da}$ (for terminal $H$ and $OH$), rounded to 2 decimals.
4. `__len__(self) -> int`:
   - Returns number of amino acids.

- **Input:** `protein_id: str`, `sequence: str`.
- **Output:** `Protein` instance.


In [ ]:
class Protein:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 3A.2
p = Protein("P01308", "GIVEQCCTSICSLYQLENYCN")  # Insulin A-chain
assert p.protein_id == "P01308"
assert len(p) == 21
assert p.molecular_weight() > 2000.0

try:
    Protein("BAD", "GIVEX")  # 'X' is invalid
    assert False, "Should raise ValueError on invalid amino acid"
except ValueError:
    pass

print("Ex 3A.2 passed ✔")


### Ex 3A.3 (★★) — Spliced `Transcript` Model

**Problem Statement:**  
Create a class `Transcript` that models an eukaryotic mRNA transcript consisting of multiple exons:
1. `__init__(self, transcript_id: str, gene_id: str, exons: list[tuple[int, int]])`:
   - `exons`: A list of `(start, end)` tuples (0-based, half-open coordinates `[start, end)`).
   - Validates that for every exon, `start < end`.
   - Stores exons sorted by start coordinate.
2. Method `spliced_length(self) -> int`:
   - Returns the sum of lengths of all exons: $\sum (end - start)$.
3. Method `exon_count(self) -> int`:
   - Returns the number of exons.
4. Method `extract_cds(self, genomic_seq: str) -> str`:
   - Slices each exon from `genomic_seq` and concatenates them into a continuous spliced CDS string.

- **Input:** Exon coordinate tuples and genomic sequence.
- **Output:** Spliced lengths and extracted CDS sequence.


In [ ]:
class Transcript:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 3A.3
# Synthetic genomic sequence: Exon 1 (0..6), Intron (6..12), Exon 2 (12..18)
synthetic_genome = "ATGCGA" + "NNNNNN" + "TTAGCT"
t = Transcript("ENST001", "ENSG001", [(12, 18), (0, 6)])  # Out of order input

assert t.exon_count() == 2
assert t.spliced_length() == 12
assert t.extract_cds(synthetic_genome) == "ATGCGATTAGCT"
print("Ex 3A.3 passed ✔")


### Ex 3A.4 (★★) — Operator Overloading for `DNASequence`

**Problem Statement:**  
Create a class `DNASequence` implementing operator overloading:
1. `__init__(self, seq_id: str, seq: str)`:
   - Stores `seq_id` and uppercase `seq`.
2. `__add__(self, other: "DNASequence") -> "DNASequence"`:
   - Concatenates two `DNASequence` objects.
   - New ID: `"{self.seq_id}_{other.seq_id}"`.
   - Returns a **new** `DNASequence` instance.
3. `__mul__(self, repeat: int) -> "DNASequence"`:
   - Implements tandem repeat multiplication: `seq * 3`.
   - New ID: `"{self.seq_id}_x{repeat}"`.
   - Returns a **new** `DNASequence` instance.
4. `__eq__(self, other: object) -> bool`:
   - Checks if sequences are equal in length and character content (case-insensitive).
5. `__len__(self) -> int`:
   - Returns sequence length.

- **Input:** `DNASequence` instances and operators (`+`, `*`, `==`, `len`).
- **Output:** Overloaded operator results.


In [ ]:
class DNASequence:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 3A.4
s1 = DNASequence("motif", "CAG")
s2 = DNASequence("tail", "TTT")

repeat = s1 * 3
assert repeat.seq == "CAGCAGCAG"
assert repeat.seq_id == "motif_x3"
assert len(repeat) == 9

joined = s1 + s2
assert joined.seq == "CAGTTT"
assert joined.seq_id == "motif_tail"

assert DNASequence("a", "cag") == DNASequence("b", "CAG")
print("Ex 3A.4 passed ✔")


### Ex 3A.5 (★★) — Factory Classmethods & Validation Staticmethods

**Problem Statement:**  
Create a class `BioRecord` with alternative constructors:
1. `__init__(self, seq_id: str, sequence: str, description: str = "")`:
   - Validates that `sequence` only contains letters `A, C, G, T, N`.
   - Stores `seq_id`, `sequence`, `description`.
2. Static method `@staticmethod validate_dna(seq_str: str) -> bool`:
   - Returns `True` if `seq_str` contains only valid DNA characters, else `False`.
3. Class method `@classmethod from_fasta_record(cls, header: str, seq: str) -> "BioRecord"`:
   - Parses `header` (which starts with `>`):
     - Extracts `seq_id` as the first whitespace-separated token after `>`.
     - Extracts remaining text as `description`.
   - Returns an initialized instance of `cls`.

- **Input:** Raw header strings and sequence strings.
- **Output:** Instantiated `BioRecord` via factory.


In [ ]:
class BioRecord:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 3A.5
assert BioRecord.validate_dna("ATGCGATC") is True
assert BioRecord.validate_dna("ATGCZ") is False

rec = BioRecord.from_fasta_record(">NM_000546.6 Homo sapiens TP53 mRNA", "ATGGAGGAGCCG")
assert rec.seq_id == "NM_000546.6"
assert rec.description == "Homo sapiens TP53 mRNA"
assert rec.sequence == "ATGGAGGAGCCG"
print("Ex 3A.5 passed ✔")


### Ex 3A.6 (★★★) — Restriction Enzyme Digestion Simulator

**Problem Statement:**  
Molecular biologists simulate restriction digestion to predict fragment sizes in gel electrophoresis.

Create a class `RestrictionEnzyme`:
1. `__init__(self, name: str, recognition_site: str, cut_offset: int)`:
   - `name`: Enzyme name (e.g. `"EcoRI"`).
   - `recognition_site`: Palindromic recognition sequence (e.g. `"GAATTC"`).
   - `cut_offset`: Index where cleavage occurs relative to the 5′ start of the recognition site (e.g. `1` for EcoRI: `G^AATTC`).
2. Method `digest(self, dna_seq: str) -> list[str]`:
   - Scans `dna_seq` (5′ to 3′) for all non-overlapping occurrences of `recognition_site`.
   - Cleaves the DNA at each site offset: cleavage index = `site_index + cut_offset`.
   - Returns a `list[str]` of all resulting fragment sequences.
   - If no cut sites are found, returns `[dna_seq]`.

- **Input:** DNA sequence string.
- **Output:** List of digested fragment strings.


In [ ]:
class RestrictionEnzyme:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 3A.6
# EcoRI recognizes G^AATTC (cut offset 1)
ecori = RestrictionEnzyme("EcoRI", "GAATTC", cut_offset=1)

# Sequence containing two EcoRI sites
# Positions: 4 (G|AATTC) and 20 (G|AATTC)
test_seq = "CCCCGAATTC" + "AAAAAAAAAA" + "GAATTCGGGG"
fragments = ecori.digest(test_seq)

print("Digestion fragments:", fragments)
assert len(fragments) == 3, f"Expected 3 fragments, got {len(fragments)}"
assert fragments[0] == "CCCCG"
assert fragments[1] == "AATTCAAAAAAAAAAG"
assert fragments[2] == "AATTCGGGG"
assert "".join(fragments) == test_seq, "Fragments must reconstitute original sequence!"
print("Ex 3A.6 passed ✔")


---
## 6. Summary & Cheat Sheet

| Feature | Syntax / Example | Purpose in Bioinformatics |
|---|---|---|
| Class Definition | `class DNASequence:` | Blueprint for biological models |
| Constructor | `def __init__(self, ...):` | Initializes state and enforces data invariants |
| Instance Attribute | `self.seq = seq` | Per-instance data (unique ID, sequence) |
| Class Attribute | `GENETIC_CODE = {...}` | Shared constants across all instances |
| Instance Method | `def gc_content(self):` | Computes metric on `self` |
| Readable String | `def __str__(self):` | User-friendly summary for `print()` |
| Developer Repr | `def __repr__(self):` | Precise technical representation |
| Length Protocol | `def __len__(self):` | Enables `len(obj)` |
| Operator Overload | `def __add__(self, other):` | Enables `seq1 + seq2` (ligation) |
| Class Method | `@classmethod def from_fasta(...)` | Alternative constructor/factory |
| Static Method | `@staticmethod def validate(...)` | Standalone utility in class namespace |

---


## 7. Further Reading & Resources

- [Real Python: Object-Oriented Programming (OOP) in Python 3](https://realpython.com/python3-object-oriented-programming/)
- [Real Python: Python's Dunder Methods (A Guide to Operator Overloading)](https://realpython.com/operator-function-overloading/)
- [PEP 8: Style Guide for Python Code (Naming Conventions)](https://peps.python.org/pep-0008/#naming-conventions)
- [Biopython `Bio.Seq` Source Code Architecture](https://github.com/biopython/biopython/tree/master/Bio/Seq.py)
